# NSRDB Data Processing

Select the required NSRDB fields, verify the source timezone from its metadata, and convert the source timestamps to Malaysia local time. The processed data remains in memory as `processed_data`.

In [33]:
import csv
from pathlib import Path

import pandas as pd

DATA_RELATIVE_PATH = Path("dataset/row/NSRDB/NSRDB_2016_2020_combined.csv")
INPUT_FILE = next(
    (
        root / DATA_RELATIVE_PATH
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / DATA_RELATIVE_PATH).is_file()
    ),
    None,
)
if INPUT_FILE is None:
    raise FileNotFoundError(f"Could not find {DATA_RELATIVE_PATH} from {Path.cwd()}")

print(f"NSRDB input file: {INPUT_FILE}")

NSRDB input file: c:\FYP\coding\dataset\row\NSRDB\NSRDB_2016_2020_combined.csv


## Verify source datetime and timezone

The NSRDB file has two metadata rows followed by its column header. Read the `Time Zone` metadata as the source UTC offset and `Local Time Zone` as the location offset. Stop rather than guessing if they do not establish the expected UTC source and Malaysia UTC+8 destination.

In [34]:
with INPUT_FILE.open("r", newline="", encoding="utf-8-sig") as source_file:
    metadata_reader = csv.reader(source_file)
    metadata_headers = next(metadata_reader)
    metadata_values = next(metadata_reader)
    source_columns = next(metadata_reader)

metadata = dict(zip(metadata_headers, metadata_values, strict=True))
source_utc_offset = float(metadata["Time Zone"])
location_utc_offset = float(metadata["Local Time Zone"])

if source_utc_offset != 0:
    raise ValueError(
        "Source timezone is not UTC (Time Zone metadata is "
        f"{metadata['Time Zone']!r}); confirm its timezone before converting."
    )
if location_utc_offset != 8:
    raise ValueError(
        "NSRDB Local Time Zone metadata does not indicate Malaysia UTC+8 "
        f"(value is {metadata['Local Time Zone']!r})."
    )

SOURCE_DATETIME_COLUMNS = ["Year", "Month", "Day", "Hour", "Minute"]
if not all(column in source_columns for column in SOURCE_DATETIME_COLUMNS):
    raise ValueError("The NSRDB datetime fields differ from the expected format.")

print(f"Original datetime fields: {', '.join(SOURCE_DATETIME_COLUMNS)}")
print("Original datetime format: separate numeric year, month, day, hour, minute fields")
print(f"Verified source timezone: UTC (metadata Time Zone={source_utc_offset:g})")
print(
    "Verified Malaysia local offset: UTC+8 "
    f"(metadata Local Time Zone={location_utc_offset:g})"
)

Original datetime fields: Year, Month, Day, Hour, Minute
Original datetime format: separate numeric year, month, day, hour, minute fields
Verified source timezone: UTC (metadata Time Zone=0)
Verified Malaysia local offset: UTC+8 (metadata Local Time Zone=8)


## Step 1: Select Required Columns

Build a timezone-aware Malaysia `Datetime` from the verified UTC source fields, sort chronologically, and retain only the requested columns in the specified order. All measurement columns are converted to `float64`.

In [35]:
TIMEZONE = "Asia/Kuala_Lumpur"
NUMERIC_COLUMNS = [
    "Temperature",
    "Clearsky DHI",
    "Clearsky DNI",
    "Clearsky GHI",
    "DHI",
    "DNI",
    "GHI",
    "Relative Humidity",
    "Solar Zenith Angle",
    "Solar Azimuth Angle",
]
OUTPUT_COLUMNS = ["Datetime", *NUMERIC_COLUMNS]
required_source_columns = [*SOURCE_DATETIME_COLUMNS, *NUMERIC_COLUMNS]
missing_columns = [
    column for column in required_source_columns if column not in source_columns
]
if missing_columns:
    raise ValueError(f"Required NSRDB columns are missing: {missing_columns}")

selected_source_data = pd.read_csv(
    INPUT_FILE,
    skiprows=2,
    usecols=required_source_columns,
    low_memory=False,
)
source_datetime = pd.to_datetime(
    {
        "year": selected_source_data["Year"],
        "month": selected_source_data["Month"],
        "day": selected_source_data["Day"],
        "hour": selected_source_data["Hour"],
        "minute": selected_source_data["Minute"],
    },
    errors="raise",
)
malaysia_datetime = source_datetime.dt.tz_localize("UTC").dt.tz_convert(TIMEZONE)

processed_data = selected_source_data[NUMERIC_COLUMNS].copy()
for column in NUMERIC_COLUMNS:
    processed_data[column] = pd.to_numeric(
        processed_data[column], errors="raise"
    ).astype("float64")
processed_data.insert(0, "Datetime", malaysia_datetime)
processed_data = (
    processed_data.sort_values("Datetime", kind="stable")
    .loc[:, OUTPUT_COLUMNS]
    .reset_index(drop=True)
)

if processed_data["Datetime"].is_monotonic_increasing is False:
    raise ValueError("Processed timestamps are not in chronological order.")
if not all(processed_data[column].dtype == "float64" for column in NUMERIC_COLUMNS):
    raise TypeError("One or more measurement columns are not float64.")

print(f"Processed rows: {len(processed_data):,}")
print(f"Output columns: {', '.join(processed_data.columns)}")
print(f"Datetime range: {processed_data['Datetime'].min()} to {processed_data['Datetime'].max()}")
print("Numerical dtypes: float64")

Processed rows: 43,848
Output columns: Datetime, Temperature, Clearsky DHI, Clearsky DNI, Clearsky GHI, DHI, DNI, GHI, Relative Humidity, Solar Zenith Angle, Solar Azimuth Angle
Datetime range: 2016-01-01 08:00:00+08:00 to 2021-01-01 07:00:00+08:00
Numerical dtypes: float64


## Step 2: Time Filtering

Keep observations from 9:00:00 through 19:00:00, inclusive, using Malaysia-local `Datetime`. The filtered result is `time_filtered_data`.

In [36]:
from datetime import time

START_TIME = time(9, 0)
END_TIME = time(19, 0)

local_times = processed_data["Datetime"].dt.time
time_filter = local_times.between(START_TIME, END_TIME, inclusive="both")
time_filtered_data = processed_data.loc[time_filter].reset_index(drop=True)

print(f"Input observations: {len(processed_data):,}")
print(f"Observations from 9:00:00 to 19:00:00 Malaysia time: {len(time_filtered_data):,}")
print(f"Included 9:00 observations: {int(local_times.eq(START_TIME).sum()):,}")
print(f"Included 19:00 observations: {int(local_times.eq(END_TIME).sum()):,}")

Input observations: 43,848
Observations from 9:00:00 to 19:00:00 Malaysia time: 20,097
Included 9:00 observations: 1,827
Included 19:00 observations: 1,827


In [37]:
OUTPUT_FILE = INPUT_FILE.parents[3] / "dataset" / "processed" / "NSRDB_processed.csv"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

time_filtered_data.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(time_filtered_data):,} rows to {OUTPUT_FILE}")

Saved 20,097 rows to c:\FYP\coding\dataset\processed\NSRDB_processed.csv
